In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import ttest_ind
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    make_scorer,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from statsmodels.stats.multitest import multipletests

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)

data_dir = next(
    (
        path
        for path in (Path("."), Path("deepgram"))
        if (path / "deepgram_dataset_with_features.csv").is_file()
    ),
    None,
)
split_dir = next(
    (
        path
        for path in (
            Path("datasets/deepgram"),
            Path("../datasets/deepgram"),
            data_dir,
        )
        if path is not None and (path / "training_422.csv").is_file()
    ),
    None,
)
if data_dir is None or split_dir is None:
    raise FileNotFoundError("Could not locate the DeepGram data files.")

combined_dataset = pd.read_csv(data_dir / "deepgram_dataset_with_features.csv")

## Standardization and statistical feature selection

In [ ]:
features = combined_dataset.drop(columns=["data", "label"])
scaled_features = StandardScaler().fit_transform(features)
scaled_dataset = pd.DataFrame(scaled_features, columns=features.columns)
scaled_dataset.insert(0, "label", combined_dataset["label"].to_numpy())
scaled_dataset.insert(0, "data", combined_dataset["data"].to_numpy())
combined_dataset = scaled_dataset

group_0 = combined_dataset.loc[combined_dataset["label"] == 0, features.columns]
group_1 = combined_dataset.loc[combined_dataset["label"] == 1, features.columns]
t_stats, p_values = ttest_ind(group_0, group_1, equal_var=False)
ttest_results = pd.DataFrame(
    {"feature": features.columns, "t_stat": t_stats, "p_value": p_values}
)

nan_features = ttest_results.loc[ttest_results["p_value"].isna(), "feature"]
combined_dataset = combined_dataset.drop(columns=nan_features.tolist())
ttest_results = ttest_results.dropna()
_, q_values, _, _ = multipletests(ttest_results["p_value"], method="fdr_bh")
ttest_results["q_value"] = q_values
significant_features = ttest_results.loc[
    ttest_results["q_value"] < 0.01, "feature"
].tolist()
cleaned_data = combined_dataset[["data", "label", *significant_features]]

## Train-validation split preparation

In [ ]:
deepgram_train = pd.concat(
    [
        pd.read_csv(split_dir / "training_422.csv"),
        pd.read_csv(split_dir / "trainingNon_422.csv"),
    ],
    ignore_index=True,
).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
deepgram_validation = pd.concat(
    [
        pd.read_csv(split_dir / "independent_150.csv"),
        pd.read_csv(split_dir / "independentNon_150.csv"),
    ],
    ignore_index=True,
).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

deepgram_train = deepgram_train.drop_duplicates(
    subset="data", keep=False
).reset_index(drop=True)
deepgram_validation = deepgram_validation.drop_duplicates(
    subset="data", keep=False
).reset_index(drop=True)

common_peptides = set(deepgram_train["data"]).intersection(
    deepgram_validation["data"]
)
train_common = deepgram_train.loc[
    deepgram_train["data"].isin(common_peptides), ["data", "label"]
].rename(columns={"label": "label_train"})
validation_common = deepgram_validation.loc[
    deepgram_validation["data"].isin(common_peptides), ["data", "label"]
].rename(columns={"label": "label_validation"})
common_labels = pd.merge(train_common, validation_common, on="data")
conflicting_peptides = common_labels.loc[
    common_labels["label_train"] != common_labels["label_validation"], "data"
].tolist()
deepgram_train = deepgram_train[
    ~deepgram_train["data"].isin(conflicting_peptides)
].reset_index(drop=True)
deepgram_validation = deepgram_validation[
    ~deepgram_validation["data"].isin(conflicting_peptides)
].reset_index(drop=True)
common_labels = common_labels[
    ~common_labels["data"].isin(conflicting_peptides)
].reset_index(drop=True)

label_zero = common_labels.loc[common_labels["label_train"] == 0, "data"].tolist()
label_one = common_labels.loc[common_labels["label_train"] == 1, "data"].tolist()
train_holdout = label_zero[:56] + label_one[: len(label_one) // 2]
validation_holdout = label_zero[56:] + label_one[len(label_one) // 2 :]
deepgram_train = deepgram_train[
    ~deepgram_train["data"].isin(train_holdout)
].reset_index(drop=True)
deepgram_validation = deepgram_validation[
    ~deepgram_validation["data"].isin(validation_holdout)
].reset_index(drop=True)

deepgram_train_df = cleaned_data[
    cleaned_data["data"].isin(deepgram_train["data"])
].copy()
deepgram_validation_df = cleaned_data[
    cleaned_data["data"].isin(deepgram_validation["data"])
].copy()

## SVM evaluation

In [ ]:
X_train = deepgram_train_df.drop(columns=["data", "label"])
y_train = deepgram_train_df["label"]
X_validation = deepgram_validation_df.drop(columns=["data", "label"])
y_validation = deepgram_validation_df["label"]

def specificity_score(y_true, y_pred):
    true_negative, false_positive, _, _ = confusion_matrix(y_true, y_pred).ravel()
    return true_negative / (true_negative + false_positive)

model = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    probability=True,
    shrinking=True,
    random_state=RANDOM_STATE,
)
scoring = {
    "accuracy": "accuracy",
    "f1": "f1",
    "precision": "precision",
    "recall": "recall",
    "roc_auc": "roc_auc",
    "mcc": make_scorer(matthews_corrcoef),
    "specificity": make_scorer(specificity_score),
}
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
cv_results = cross_validate(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=1,
)

def mean_std(scores):
    return f"{np.mean(scores):.4f} +/- {np.std(scores):.4f}"

print("10-Fold CV Results (Train Set - SVM)")
print(f"Accuracy     : {mean_std(cv_results['test_accuracy'])}")
print(f"F1 Score     : {mean_std(cv_results['test_f1'])}")
print(f"Sensitivity  : {mean_std(cv_results['test_recall'])}")
print(f"Specificity  : {mean_std(cv_results['test_specificity'])}")
print(f"Precision    : {mean_std(cv_results['test_precision'])}")
print(f"AUC (ROC)    : {mean_std(cv_results['test_roc_auc'])}")
print(f"MCC          : {mean_std(cv_results['test_mcc'])}")

model.fit(X_train, y_train)
y_pred = model.predict(X_validation)
y_probability = model.predict_proba(X_validation)[:, 1]
true_negative, false_positive, _, _ = confusion_matrix(
    y_validation, y_pred
).ravel()
print("\nValidation Set Results (SVM)")
print(f"Accuracy     : {accuracy_score(y_validation, y_pred):.4f}")
print(f"F1 Score     : {f1_score(y_validation, y_pred):.4f}")
print(f"Sensitivity  : {recall_score(y_validation, y_pred):.4f}")
print(
    f"Specificity  : {true_negative / (true_negative + false_positive):.4f}"
)
print(f"Precision    : {precision_score(y_validation, y_pred):.4f}")
print(f"AUC (ROC)    : {roc_auc_score(y_validation, y_probability):.4f}")
print(f"MCC          : {matthews_corrcoef(y_validation, y_pred):.4f}")